# Dense Matrices Benchmark: LU vs Matrix Inverse vs QR

This notebook is based on the **dense matrices** section of <File>Poster___numerical_linear_algebra (3).pdf</File>.

From the poster:
- LU is reported as the fastest method for **general dense matrices** over the tested sizes.
- The tested sizes are **n = 50, 100, 200, 400, 800**.
- Matrix inverse is described as consistently slower than LU, and QR becomes relatively slower as `n` increases.

Because the poster does **not provide the raw dense timing table**, this notebook reproduces the **experiment setup** for the same `n` values rather than reconstructing exact poster points.


## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter
from scipy.linalg import lu_factor, lu_solve

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.6g}')


## Matrix generator

General dense matrix generator for the same `n` across all three methods.


In [ ]:
def generate_general_dense(n: int, rng: np.random.Generator):
    """Create a random dense linear system A x = b."""
    A = rng.standard_normal((n, n)) + 5.0 * np.eye(n)
    b = rng.standard_normal(n)
    return A, b


## Solvers


In [ ]:
def solve_lu(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    lu, piv = lu_factor(A, check_finite=False)
    return lu_solve((lu, piv), b, check_finite=False)

def solve_matrix_inverse(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.linalg.inv(A) @ b

def solve_qr(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    Q, R = np.linalg.qr(A)
    return np.linalg.solve(R, Q.T @ b)

METHODS = {
    'LU': solve_lu,
    'MatrixInverse': solve_matrix_inverse,
    'QR': solve_qr,
}


## Benchmark helpers

Each method is timed on the **same matrix size `n`**.

For each `n`, the benchmark:
- generates several realizations of `A, b`,
- runs all three solvers on the same realization,
- records mean timing and simple accuracy diagnostics.


In [ ]:
def benchmark_solver(solver, A: np.ndarray, b: np.ndarray, repeats: int = 5, warmups: int = 1):
    for _ in range(warmups):
        solver(A, b)

    timings = []
    x_last = None
    for _ in range(repeats):
        t0 = perf_counter()
        x_last = solver(A, b)
        timings.append(perf_counter() - t0)

    x_ref = np.linalg.solve(A, b)
    rel_residual = np.linalg.norm(A @ x_last - b) / np.linalg.norm(b)
    rel_error_vs_ref = np.linalg.norm(x_last - x_ref) / np.linalg.norm(x_ref)

    return {
        'mean_s': float(np.mean(timings)),
        'std_s': float(np.std(timings, ddof=1)) if len(timings) > 1 else 0.0,
        'min_s': float(np.min(timings)),
        'max_s': float(np.max(timings)),
        'rel_residual': float(rel_residual),
        'rel_error_vs_ref': float(rel_error_vs_ref),
    }

def run_dense_benchmark(dimensions=(50, 100, 200, 400, 800), realizations=3, repeats=3, seed=42):
    rng = np.random.default_rng(seed)
    rows = []

    for n in dimensions:
        for trial in range(realizations):
            A, b = generate_general_dense(n, rng)

            for method_name, solver in METHODS.items():
                stats = benchmark_solver(solver, A, b, repeats=repeats)
                rows.append({
                    'n': n,
                    'trial': trial,
                    'Method': method_name,
                    **stats,
                })

    return pd.DataFrame(rows)


## Run the experiment

These are the same `n` values highlighted in the poster's dense-matrix finding.


In [ ]:
dimensions = (50, 100, 200, 400, 800)
results = run_dense_benchmark(dimensions=dimensions, realizations=3, repeats=3, seed=42)
results.head()


## Aggregate summary


In [ ]:
summary = (results
           .groupby(['n', 'Method'], as_index=False)
           .agg(mean_time_s=('mean_s', 'mean'),
                std_time_s=('mean_s', 'std'),
                mean_rel_residual=('rel_residual', 'mean'),
                mean_rel_error_vs_ref=('rel_error_vs_ref', 'mean')))
summary


## Pivoted comparison table


In [ ]:
pivot = summary.pivot(index='n', columns='Method', values='mean_time_s').reset_index()
pivot['Inverse_vs_LU'] = pivot['MatrixInverse'] / pivot['LU']
pivot['QR_vs_LU'] = pivot['QR'] / pivot['LU']
pivot


## Plot: execution time vs matrix size

This is the core dense-matrix plot: **LU, Matrix Inverse, QR** on the same `n`.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.2))

for method, marker in [('LU', 'o'), ('MatrixInverse', 's'), ('QR', '^')]:
    sub = summary[summary['Method'] == method].sort_values('n')
    ax.errorbar(
        sub['n'], sub['mean_time_s'], yerr=sub['std_time_s'],
        fmt=f'{marker}-', lw=2, ms=7, capsize=4, label=method
    )

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Matrix dimension (n)')
ax.set_ylabel('Execution time (seconds)')
ax.set_title('Dense matrices: LU vs Matrix Inverse vs QR')
ax.legend()
plt.tight_layout()
plt.show()


## Plot: slowdown relative to LU

This makes it easy to compare with the poster's narrative claims.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.2))
ax.semilogx(pivot['n'], pivot['Inverse_vs_LU'], 's-', lw=2, ms=7, label='Matrix Inverse / LU')
ax.semilogx(pivot['n'], pivot['QR_vs_LU'], '^-', lw=2, ms=7, label='QR / LU')
ax.axhline(1.0, color='black', ls='--', lw=1)
ax.set_xlabel('Matrix dimension (n)')
ax.set_ylabel('Slowdown factor relative to LU')
ax.set_title('Relative slowdown compared with LU')
ax.legend()
plt.tight_layout()
plt.show()


## Optional: estimate scaling exponent

Fits a power law `time ≈ C * n^p` on the measured results.


In [ ]:
def fit_power_law(x, y):
    p, log_c = np.polyfit(np.log(x), np.log(y), 1)
    return p, np.exp(log_c)

for method in ['LU', 'MatrixInverse', 'QR']:
    sub = summary[summary['Method'] == method].sort_values('n')
    p, c = fit_power_law(sub['n'].to_numpy(), sub['mean_time_s'].to_numpy())
    print(f'{method:13s}  exponent p = {p:.3f}, constant C = {c:.3e}')


## Interpretation template

Use this cell after running the notebook:

- Identify which method is fastest at each `n`.
- Compare `MatrixInverse / LU` and `QR / LU` with the poster's qualitative findings.
- Note that exact timings will depend on machine, BLAS/LAPACK backend, and runtime environment.
